# AI应用基础 · 工单M4.1《工业故障报告自动分类系统》
**第10周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 混淆矩阵分析哪两类易混并给出词表级解释（如液压vs传动共享词）；示范讲解 |
| **B 组长** | 标准达成 | ≥0.85；混淆矩阵分析+关键词解释 |
| **C 操作员** | 基础完成 | 准确率≥0.75；能预测新文本 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：TF-IDF中的IDF让什么样的词权重更高？停用词不滤会怎样？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4"], check=True)
print("数据已生成: data/")

工作目录: D:\Python编程实战\教学实施资料\01-AI应用基础
数据已生成: data/


## 三、任务要求（Make）
**实施步骤**
1. jieba分词+停用词过滤
2. TF-IDF向量化(2000维)
3. 训练两分类器并对比指标
4. 预测自收集新文本+Top10关键词

**验收标准**：及格(D)=`准确率≥0.75；能预测新文本` ；优秀(B+)=`≥0.85；混淆矩阵分析+关键词解释`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
# -*- coding: utf-8 -*-
"""M4.1 工业故障报告自动分类 —— 学生代码框架"""
import csv

import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

DATA = "data/fault_reports.csv"
STOP = set("的 了 在 是 请 尽快 处理 影响 生产 班中 巡检 发现 已 挂牌 停机".split())

### Starter 2：`def load():`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
def load():
    texts, labels = [], []
    with open(DATA, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(row["label"])
    return texts, labels

### Starter 3：`def cut(text):`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
def cut(t):
    """jieba分词+去停用词, 空格连接"""
    import jieba
    stop = set("的 了 在 是 请 尽快 处理 影响 生产 班中 巡检 发现 已 挂牌 停机".split())
    return " ".join(w for w in jieba.cut(t) if w not in stop and w.strip())

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-2: fit(tr_x, tr_y) → predict(te_x)
>   → 记录accuracy
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-3: 新文本预测(自收集的2条+示例)
> - TODO-4: 每类Top10关键词(pipe[:-
>   1].get_feature_names_out +
>   coef/feature_log_prob_) TODO-5:
>   (优秀)混淆矩阵: 哪两类易混? 为什么?

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`准确率≥0.75；能预测新文本`，优秀(B+)=`≥0.85；混淆矩阵分析+关键词解释`

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
def main():
    texts, labels = load()
    corpus = [cut(t) for t in texts]
    n = int(len(corpus) * 0.8)
    tr_x, te_x = corpus[:n], corpus[n:]
    tr_y, te_y = labels[:n], labels[n:]

    results = {}
    for name, clf in [("朴素贝叶斯", MultinomialNB()),
                      ("逻辑回归", LogisticRegression(max_iter=1000))]:
        pipe = make_pipeline(TfidfVectorizer(max_features=2000), clf)
        # TODO-2: fit(tr_x, tr_y) → predict(te_x) → 记录accuracy
        # 注意: TfidfVectorizer只在训练集fit(Pipeline已保证)
        # === 你的代码 ===
        # results[name] = acc
        # print(classification_report(te_y, pred))

    # TODO-3: 新文本预测(自收集的2条+示例)
    # for q in ["3号注塑机液压油温升高报警", "包装线PLC从站掉线"]:
    #     print(q, "->", pipe.predict([cut(q)]))

    # TODO-4: 每类Top10关键词(pipe[:-1].get_feature_names_out + coef/feature_log_prob_)
    # TODO-5: (优秀)混淆矩阵: 哪两类易混? 为什么?

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M4.1.py` 或本notebook(含输出)
- ⚠ 停用词不滤则“的/了”霸榜Top10
- ⚠ Pipeline组合向量化与分类器可自动避免泄漏
- ⚠ 新文本预测要过同一个cut()预处理函数

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m41()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| `main()` | 见下方 `m41()` 内对应逻辑 |

In [1]:
def m41():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, classification_report, f1_score
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import make_pipeline
    import jieba

    texts, labels = [], []
    with open("data/fault_reports.csv", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            texts.append(r["text"])
            labels.append(r["label"])
    stop = set("的 了 在 是 请 尽快 处理 影响 生产 班中 巡检 发现 已 挂牌 停机".split())

    def cut(t):
        return " ".join(w for w in jieba.cut(t) if w not in stop and w.strip())

    corpus = [cut(t) for t in texts]
    n = int(len(corpus) * 0.8)
    print("=== 工单 M4.1 故障报告文本分类 · 完成态 ===")
    print(f"语料: data/fault_reports.csv  {len(texts)} 条 · {len(set(labels))} 类"
          f"（{'/'.join(sorted(set(labels)))}）")
    print(f"流水线: jieba分词+停用词 → TF-IDF → 线性分类器　划分 训练{n}/测试{len(corpus) - n}")
    rows = []
    for name, clf in [("NB(朴素贝叶斯)", MultinomialNB()),
                      ("LR(逻辑回归)", LogisticRegression(max_iter=1000))]:
        pipe = make_pipeline(TfidfVectorizer(max_features=2000), clf)
        pipe.fit(corpus[:n], labels[:n])
        pred = pipe.predict(corpus[n:])
        acc = accuracy_score(labels[n:], pred)
        f1 = f1_score(labels[n:], pred, average="macro")
        rows.append((name, acc, f1))
        print(f"  {name:<14} accuracy={acc:.3f}   macro-F1={f1:.3f}")
    best = max(rows, key=lambda r: r[2])
    print(f"对比结论: {best[0]} macro-F1 最高（{best[2]:.3f}）—— 短文本上线性模型已足够")
    demo = "3号注塑机液压油温升高报警"
    pipe2 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    pipe2.fit(corpus, labels)
    demo_pred = pipe2.predict([cut(demo)])[0]
    print(f"在线演示: 「{demo}」→ 预测类别「{demo_pred}」")
    print("结论: TF-IDF+线性模型即可高精度归类工业短文本；LLM 的价值在开放问答与生成解释"
          "（见 M4.2 RAG），二者互补而非替代。")

    # ---- 明细（供工单/答辩附录，不进PPT效果图） ----
    print("\n[明细] 分类报告（测试集）:")
    pipe = make_pipeline(TfidfVectorizer(max_features=2000), LogisticRegression(max_iter=1000))
    pipe.fit(corpus[:n], labels[:n])
    print(classification_report(labels[n:], pipe.predict(corpus[n:])))

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。